<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/TN-CNN-02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import pandas as pd

df = pd.read_csv("HPV2026.csv")

print(df.shape)

print(df.columns.tolist())

df.head()

(782, 13)
['PatientID', 'CenterID', 'Age', 'Gender', 'Tobacco Consumption', 'Alcohol Consumption', 'Performance Status', 'Treatment', 'HPV Status', 'Relapse', 'RFS', 'T-stage', 'N-stage']


,PatientID,CenterID,Age,Gender,Tobacco Consumption,Alcohol Consumption,Performance Status,Treatment,HPV Status,Relapse,RFS,T-stage,N-stage
0,CHUM-001,1.0,82.0,1.0,NaN,NaN,NaN,1.0,NaN,0.0,1704.0,T2,N2
1,CHUM-002,1.0,73.0,1.0,NaN,NaN,NaN,1.0,NaN,1.0,439.0,T3,N1
2,CHUM-006,1.0,65.0,1.0,NaN,NaN,NaN,1.0,NaN,0.0,1186.0,T2,N2
3,CHUM-007,1.0,70.0,0.0,NaN,NaN,NaN,0.0,NaN,0.0,1702.0,T2,N2
4,CHUM-008,1.0,67.0,0.0,NaN,NaN,NaN,1.0,NaN,0.0,1499.0,T2,N2


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PatientID            782 non-null    object 
 1   CenterID             782 non-null    float64
 2   Age                  782 non-null    float64
 3   Gender               782 non-null    float64
 4   Tobacco Consumption  522 non-null    float64
 5   Alcohol Consumption  516 non-null    float64
 6   Performance Status   468 non-null    float64
 7   Treatment            763 non-null    float64
 8   HPV Status           597 non-null    float64
 9   Relapse              727 non-null    float64
 10  RFS                  727 non-null    float64
 11  T-stage              778 non-null    object 
 12  N-stage              782 non-null    object 
dtypes: float64(10), object(3)
memory usage: 79.6+ KB


In [49]:
print(df["HPV Status"].value_counts())

HPV Status
1.0    533
0.0     64
Name: count, dtype: int64


In [50]:
print(df["HPV Status"].isna().sum())

185


In [51]:
df_train = df.dropna(subset=["HPV Status"])

print(df_train.shape)

(597, 13)


In [52]:
print(df_train.isnull().sum())

PatientID                0
CenterID                 0
Age                      0
Gender                   0
Tobacco Consumption     99
Alcohol Consumption    100
Performance Status     146
Treatment               19
HPV Status               0
Relapse                 43
RFS                     43
T-stage                  3
N-stage                  0
dtype: int64


In [53]:
df_tn = df.dropna(
    subset=[
        "T-stage",
        "N-stage"
    ]
).copy()

print(df_tn.shape)

(778, 13)


In [54]:
print(df_tn["T-stage"].value_counts())

T-stage
T2    285
T1    189
T3    183
T4    118
T0      3
Name: count, dtype: int64


In [55]:
print(df_tn["N-stage"].value_counts())

N-stage
N2    468
N3    147
N0     87
N1     76
Name: count, dtype: int64


In [56]:
print(df_tn.isnull().sum())

PatientID                0
CenterID                 0
Age                      0
Gender                   0
Tobacco Consumption    259
Alcohol Consumption    265
Performance Status     313
Treatment               19
HPV Status             184
Relapse                 55
RFS                     55
T-stage                  0
N-stage                  0
dtype: int64


In [57]:
df_tn["T-stage"] = df_tn["T-stage"].replace({
    "T1":0,
    "T2":1,
    "T3":2,
    "T4":3
})

In [58]:
df_tn["N-stage"] = df_tn["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

/tmp/ipykernel_656/2704620313.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_tn["N-stage"] = df_tn["N-stage"].replace({


In [59]:
print(df_tn["T-stage"].value_counts())

T-stage
1     285
0     189
2     183
3     118
T0      3
Name: count, dtype: int64


In [60]:
df_tn = df_tn[df_tn["T-stage"] != "T0"]

In [61]:
print(df_tn["T-stage"].value_counts())

T-stage
1    285
0    189
2    183
3    118
Name: count, dtype: int64


In [62]:
print(df_tn["T-stage"].dtype)
print(df_tn["N-stage"].dtype)

object
int64


In [63]:
df_tn["T-stage"] = df_tn["T-stage"].astype(int)

In [64]:
print(df_tn["T-stage"].dtype)
print(df_tn["N-stage"].dtype)

int64
int64


In [65]:
print(df_tn.isnull().sum())

PatientID                0
CenterID                 0
Age                      0
Gender                   0
Tobacco Consumption    259
Alcohol Consumption    265
Performance Status     313
Treatment               19
HPV Status             184
Relapse                 55
RFS                     55
T-stage                  0
N-stage                  0
dtype: int64


In [66]:
tobacco_mode = df_tn["Tobacco Consumption"].mode()[0]

df_tn["Tobacco Consumption"] = (
    df_tn["Tobacco Consumption"]
    .fillna(tobacco_mode)
)

In [67]:
alcohol_mode = df_tn["Alcohol Consumption"].mode()[0]

df_tn["Alcohol Consumption"] = (
    df_tn["Alcohol Consumption"]
    .fillna(alcohol_mode)
)

In [68]:
performance_mode = df_tn["Performance Status"].mode()[0]

df_tn["Performance Status"] = (
    df_tn["Performance Status"]
    .fillna(performance_mode)
)

In [69]:
treatment_mode = df_tn["Treatment"].mode()[0]

df_tn["Treatment"] = (
    df_tn["Treatment"]
    .fillna(treatment_mode)
)

In [70]:
print(df_tn.isnull().sum())

PatientID                0
CenterID                 0
Age                      0
Gender                   0
Tobacco Consumption      0
Alcohol Consumption      0
Performance Status       0
Treatment                0
HPV Status             184
Relapse                 55
RFS                     55
T-stage                  0
N-stage                  0
dtype: int64


In [71]:
X = df_tn[
    [
        "Age",
        "Gender",
        "Tobacco Consumption",
        "Alcohol Consumption",
        "Performance Status",
        "Treatment"
    ]
]

In [72]:
y_t = df_tn["T-stage"]

y_n = df_tn["N-stage"]

In [73]:
print(X.shape)

print(y_t.shape)

print(y_n.shape)

(775, 6)
(775,)
(775,)


In [74]:
SEED = 44

In [75]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_t_train, y_t_test, y_n_train, y_n_test = train_test_split(
    X,
    y_t,
    y_n,
    test_size=0.20,
    random_state=SEED
)

In [76]:
print(X_train.shape)

print(X_test.shape)

(620, 6)
(155, 6)


In [77]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [78]:
X_train_final, X_val, \
y_t_train_final, y_t_val, \
y_n_train_final, y_n_val = train_test_split(
    X_train,
    y_t_train,
    y_n_train,
    test_size=0.20,
    random_state=SEED
)

In [79]:
print(X_train_final.shape)

print(X_val.shape)

print(X_test.shape)

(496, 6)
(124, 6)
(155, 6)


In [80]:
import torch

X_train_final = torch.FloatTensor(X_train_final)

X_val = torch.FloatTensor(X_val)

X_test = torch.FloatTensor(X_test)

y_t_train_final = torch.LongTensor(
    y_t_train_final.to_numpy()
)

y_t_val = torch.LongTensor(
    y_t_val.to_numpy()
)

y_t_test = torch.LongTensor(
    y_t_test.to_numpy()
)

y_n_train_final = torch.LongTensor(
    y_n_train_final.to_numpy()
)

y_n_val = torch.LongTensor(
    y_n_val.to_numpy()
)

y_n_test = torch.LongTensor(
    y_n_test.to_numpy()
)

In [81]:
print(X_train_final.shape)

print(X_val.shape)

print(X_test.shape)

print(y_t_train_final.shape)

print(y_n_train_final.shape)

torch.Size([496, 6])
torch.Size([124, 6])
torch.Size([155, 6])
torch.Size([496])
torch.Size([496])


In [82]:
X_train_final = X_train_final.unsqueeze(1)

X_val = X_val.unsqueeze(1)

X_test = X_test.unsqueeze(1)

print(X_train_final.shape)

print(X_val.shape)

print(X_test.shape)

torch.Size([496, 1, 6])
torch.Size([124, 1, 6])
torch.Size([155, 1, 6])


In [83]:
import torch.nn as nn

class TNCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(
            16 * 6,
            32
        )

        self.fc2 = nn.Linear(
            32,
            16
        )

        self.t_head = nn.Linear(
            16,
            4
        )

        self.n_head = nn.Linear(
            16,
            4
        )

    def forward(self, x):

        x = self.relu(
            self.conv1(x)
        )

        x = self.flatten(x)

        x = self.relu(
            self.fc1(x)
        )

        x = self.relu(
            self.fc2(x)
        )

        t_out = self.t_head(x)

        n_out = self.n_head(x)

        return t_out, n_out

In [84]:
model = TNCNN()

print(model)

TNCNN(
  (conv1): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=96, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (t_head): Linear(in_features=16, out_features=4, bias=True)
  (n_head): Linear(in_features=16, out_features=4, bias=True)
)


In [85]:
t_weights = torch.tensor(
    [1.5, 1.0, 1.5, 2.0],
    dtype=torch.float32
)

n_weights = torch.tensor(
    [2.0, 2.0, 1.0, 1.5],
    dtype=torch.float32
)

criterion_t = nn.CrossEntropyLoss(
    weight=t_weights
)

criterion_n = nn.CrossEntropyLoss(
    weight=n_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

print(criterion_t)

print(criterion_n)

print(optimizer)

CrossEntropyLoss()
CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [86]:
epochs = 500

best_val_loss = float('inf')

best_epoch = 0

for epoch in range(epochs):

    model.train()

    pred_t, pred_n = model(
        X_train_final
    )

    loss_t = criterion_t(
        pred_t,
        y_t_train_final
    )

    loss_n = criterion_n(
        pred_n,
        y_n_train_final
    )

    train_loss = (
        loss_t +
        loss_n
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_t, val_n = model(
            X_val
        )

        val_loss_t = criterion_t(
            val_t,
            y_t_val
        )

        val_loss_n = criterion_n(
            val_n,
            y_n_val
        )

        val_loss = (
            val_loss_t +
            val_loss_n
        )

    if val_loss.item() < best_val_loss:

        best_val_loss = val_loss.item()

        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_tn_cnn_weighted.pth"
        )

    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Val={val_loss.item():.4f}"
        )

print()

print(
    "Best Validation Loss =",
    best_val_loss
)

print(
    "Best Epoch =",
    best_epoch
)

Epoch 50, Train=2.7207, Val=2.7069
Epoch 100, Train=2.6941, Val=2.6883
Epoch 150, Train=2.6643, Val=2.6715
Epoch 200, Train=2.6309, Val=2.6583
Epoch 250, Train=2.5974, Val=2.6472
Epoch 300, Train=2.5644, Val=2.6370
Epoch 350, Train=2.5334, Val=2.6285
Epoch 400, Train=2.5052, Val=2.6240
Epoch 450, Train=2.4797, Val=2.6228
Epoch 500, Train=2.4579, Val=2.6229

Best Validation Loss = 2.6226723194122314
Best Epoch = 461


In [87]:
model.load_state_dict(
    torch.load("best_tn_cnn_weighted.pth")
)

<All keys matched successfully>

In [88]:
model.eval()

with torch.no_grad():

    pred_t, pred_n = model(
        X_test
    )

    pred_t_class = torch.argmax(
        pred_t,
        dim=1
    )

    pred_n_class = torch.argmax(
        pred_n,
        dim=1
    )

In [89]:
from sklearn.metrics import classification_report

print("T-STAGE")

print(
    classification_report(
        y_t_test.numpy(),
        pred_t_class.numpy()
    )
)

T-STAGE
              precision    recall  f1-score   support

           0       0.47      0.58      0.52        33
           1       0.39      0.20      0.26        60
           2       0.33      0.40      0.36        43
           3       0.16      0.26      0.20        19

    accuracy                           0.34       155
   macro avg       0.34      0.36      0.33       155
weighted avg       0.36      0.34      0.34       155



In [90]:
print("N-STAGE")

print(
    classification_report(
        y_n_test.numpy(),
        pred_n_class.numpy()
    )
)

N-STAGE
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        18
           1       0.00      0.00      0.00        18
           2       0.59      0.97      0.73        92
           3       0.00      0.00      0.00        27

    accuracy                           0.57       155
   macro avg       0.15      0.24      0.18       155
weighted avg       0.35      0.57      0.43       155



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [91]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        y_t_test.numpy(),
        pred_t_class.numpy()
    )
)

[[19  6  6  2]
 [14 12 22 12]
 [ 4  9 17 13]
 [ 3  4  7  5]]


In [92]:
print(
    confusion_matrix(
        y_n_test.numpy(),
        pred_n_class.numpy()
    )
)

[[ 0  0 18  0]
 [ 0  0 18  0]
 [ 3  0 89  0]
 [ 0  0 27  0]]
